**Install Feast**

In [ ]:
!pip -q install feast==0.64.0 pyarrow

In [ ]:
import feast

print("Feast version:", feast.__version__)

**Upload the Skill Gap Dataset**

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import pandas as pd

df = pd.read_csv("CSE_Skill_Gap_Dataset_1000.csv")

print(df.head())
print(df.shape)


**Examine the Data**

In [ ]:
print(df.columns)
print(df.info())
print(df.isnull().sum())


**Create Features- perform simple feature engineering.**

In [ ]:
import pandas as pd

# Entity
df["student_id"] = range(1, len(df) + 1)

# Handle missing values
numeric_columns = [
    "Programming", "DataStructures", "Algorithms",
    "Database", "WebDevelopment", "Cloud", "AI_ML",
    "Communication", "ProblemSolving", "Teamwork", "Projects",
    "Internship", "Certifications", "CurriculumAlignment",
    "IndustryRequirement", "SkillGap"
]

for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())

# Rename / cast features
# 'CGPA' column does not exist, removing this line
df["programming"] = df["Programming"].astype("int64")
df["data_structures"] = df["DataStructures"].astype("int64")
df["algorithms"] = df["Algorithms"].astype("int64")
df["database"] = df["Database"].astype("int64")
df["web_development"] = df["WebDevelopment"].astype("int64")
df["cloud"] = df["Cloud"].astype("int64")
df["ai_ml"] = df["AI_ML"].astype("int64")
df["communication"] = df["Communication"].astype("int64")
df["problem_solving"] = df["ProblemSolving"].astype("int64")
df["teamwork"] = df["Teamwork"].astype("int64")
df["projects"] = df["Projects"].astype("int64")
df["internship"] = df["Internship"].astype("int64")
df["certifications"] = df["Certifications"].astype("int64")
df["curriculum_alignment"] = df["CurriculumAlignment"].astype("int64")
df["industry_requirement"] = df["IndustryRequirement"].astype("int64")
df["skill_gap"] = df["SkillGap"].astype("int64")

# Derived feature 1: average technical skill
df["technical_skill_avg"] = (
    df["programming"] + df["data_structures"] + df["algorithms"]
    + df["database"] + df["web_development"] + df["cloud"] + df["ai_ml"]
) / 7
df["technical_skill_avg"] = df["technical_skill_avg"].astype("float32")

# Derived feature 2: average professional skill
df["professional_skill_avg"] = (
    df["communication"] + df["problem_solving"] + df["teamwork"]
) / 3
df["professional_skill_avg"] = df["professional_skill_avg"].astype("float32")

# Target
df["placement_status"] = df["GapStatus"].astype("int64")

In [ ]:
df[
    [
        "student_id", "programming", "data_structures",
        "algorithms", "database", "web_development", "cloud", "ai_ml",
        "communication", "problem_solving", "teamwork", "projects",
        "internship", "certifications", "curriculum_alignment",
        "industry_requirement", "skill_gap",
        "technical_skill_avg", "professional_skill_avg",
        "placement_status"
    ]
].head()

**Add Event Timestamps**

Feast models feature values with timestamps so that historical retrieval can perform point-in-time joins. The Skill Gap dataset is not naturally a time-series dataset, so artificial timestamps are created for this classroom demonstration.


**Separate Features and Labels**

In [ ]:
import pandas as pd

# Add event timestamps (moved from cell E7uo_70r0fIb to ensure columns exist)
base_time = pd.Timestamp(
    "2026-01-01",
    tz="UTC"
)

df["event_timestamp"] = (
    base_time + pd.to_timedelta(df["student_id"], unit="s")
)

df["created_timestamp"] = (
    df["event_timestamp"] + pd.Timedelta(seconds=1)
)

feature_df = df[
    [
        "student_id", "event_timestamp", "created_timestamp",
        "programming", "data_structures", "algorithms",
        "database", "web_development", "cloud", "ai_ml",
        "communication", "problem_solving", "teamwork", "projects",
        "internship", "certifications", "curriculum_alignment",
        "industry_requirement", "skill_gap",
        "technical_skill_avg", "professional_skill_avg"
    ]
].copy()

In [ ]:
label_df = df[
    ["student_id", "event_timestamp", "placement_status"]
].copy()


In [ ]:
print("Feature data:")
display(feature_df.head())

print("Labels:")
display(label_df.head())

**Create the Feast Repository**

In [ ]:
import os

repo_path = "/content/skill_gap_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)


In [ ]:
import os

repo_path = "/content/skill_gap_feast"

os.makedirs(
    f"{repo_path}/data",
    exist_ok=True
)


In [ ]:
feature_df.to_parquet(
    f"{repo_path}/data/skill_gap_features.parquet",
    index=False
)


Create feature_store.yaml

In [ ]:
feature_store_yaml = """
project: skill_gap_project

registry: data/registry.db

provider: local

offline_store:
  type: file

online_store:
  type: sqlite
  path: data/online_store.db
"""

with open(
    f"{repo_path}/feature_store.yaml",
    "w"
) as f:
    f.write(feature_store_yaml)


**Define the Entity and Feature View** Create:features.py

In [ ]:
feature_definition = """
from datetime import timedelta

from feast import (
    Entity,
    FeatureView,
    FeatureService,
    Field,
    FileSource
)

from feast.types import (
    Float32,
    Int64
)

student = Entity(
    name="student",
    join_keys=["student_id"],
    description="CSE student"
)

skill_gap_source = FileSource(
    name="skill_gap_source",
    path="data/skill_gap_features.parquet",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp"
)

skill_gap_feature_view = FeatureView(
    name="skill_gap_features",
    entities=[student],
    ttl=timedelta(days=50000),
    schema=[
        Field(name="programming", dtype=Int64),
        Field(name="data_structures", dtype=Int64),
        Field(name="algorithms", dtype=Int64),
        Field(name="database", dtype=Int64),
        Field(name="web_development", dtype=Int64),
        Field(name="cloud", dtype=Int64),
        Field(name="ai_ml", dtype=Int64),
        Field(name="communication", dtype=Int64),
        Field(name="problem_solving", dtype=Int64),
        Field(name="teamwork", dtype=Int64),
        Field(name="projects", dtype=Int64),
        Field(name="internship", dtype=Int64),
        Field(name="certifications", dtype=Int64),
        Field(name="curriculum_alignment", dtype=Int64),
        Field(name="industry_requirement", dtype=Int64),
        Field(name="skill_gap", dtype=Int64),
        Field(name="technical_skill_avg", dtype=Float32),
        Field(name="professional_skill_avg", dtype=Float32),
    ],
    source=skill_gap_source,
    online=True
)

skill_gap_feature_service = FeatureService(
    name="skill_gap_placement_service",
    features=[skill_gap_feature_view]
)
"""

with open(
    f"{repo_path}/features.py",
    "w"
) as f:
    f.write(feature_definition)

In [ ]:
!find /content/skill_gap_feast -maxdepth 2 -type f


**Apply the Feature Definitions**

In [ ]:
%cd /content/skill_gap_feast


In [ ]:
!feast apply

In [ ]:
!feast entities list

In [ ]:
!feast feature-views list

**Create the Feast Store Object**

In [ ]:
from feast import FeatureStore

store = FeatureStore(
    repo_path="/content/skill_gap_feast"
)


In [ ]:
feature_service = store.get_feature_service(
    "skill_gap_placement_service"
)


**Retrieve Historical Features**

In [ ]:
entity_df = label_df.copy()

display(entity_df.head())

In [ ]:
training_data = store.get_historical_features(
    entity_df=entity_df,
    features=feature_service
).to_df()

In [ ]:
display(training_data.head())

**Train a Model Using Feast Features**

In [ ]:
feature_columns = [
    "programming", "data_structures", "algorithms",
    "database", "web_development", "cloud", "ai_ml",
    "communication", "problem_solving", "teamwork", "projects",
    "internship", "certifications", "curriculum_alignment",
    "industry_requirement", "skill_gap",
    "technical_skill_avg", "professional_skill_avg"
]

In [ ]:
X = training_data[feature_columns]
y = training_data["placement_status"]


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(
    X_train,
    y_train
)

In [ ]:
predictions = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(
    y_test,
    predictions
)

print(
    "Accuracy:",
    round(accuracy * 100, 2),
    "%"
)

The important point is not the accuracy. The model received its feature matrix through Feast historical feature retrieval.

**Materialize Features to the Online Store**

In [ ]:
%cd /content/skill_gap_feast

!feast materialize \
    2026-01-01T00:00:00 \
    2026-01-02T00:00:00


**Retrieve Online Features**

In [ ]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"student_id": 25}
    ]
).to_dict()


In [ ]:
print(online_features)

In [ ]:
online_df = pd.DataFrame(
    online_features
)

display(online_df)

**Make an Online Prediction**

In [ ]:
final_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

final_model.fit(X, y)

In [ ]:
X_online = online_df[
    feature_columns
]

In [ ]:
prediction = final_model.predict(
    X_online
)

print(
    "Predicted placement status:",
    prediction[0]
)


**Retrieve Features for Multiple Students**

In [ ]:
online_features = store.get_online_features(
    features=feature_service,
    entity_rows=[
        {"student_id": 10},
        {"student_id": 20},
        {"student_id": 30},
        {"student_id": 40}
    ]
).to_dict()

online_df = pd.DataFrame(
    online_features
)

display(online_df)


In [ ]:
online_df["predicted_placement_status"] = (
    final_model.predict(
        online_df[feature_columns]
    )
)

display(
    online_df[
        [
            "student_id",
            "predicted_placement_status"
        ]
    ]
)
